# Tanarang data pipeline

Disclaimer: The copyright of data from www.tanarang.com belongs to Tanarang.
This is being used purely for academic, non-commercial purposes.

## Imports

In [ ]:
import os
import csv
import pandas as pd
import requests
from bs4 import BeautifulSoup
import dill
import json

In [ ]:
from libmogra.datatypes import SSwar
from libmogra.raagdb.tanarang_parser import TanarangParsedRaag

In [ ]:
SCRIPT_DIR = os.path.abspath("")
TEMP_DIR = os.path.join(SCRIPT_DIR, "temp")

os.makedirs(TEMP_DIR, exist_ok=True)

## Vars

In [ ]:
RECONSTRUCT_FROM_SCRATCH = False

In [ ]:
raag_list_path = os.path.join(TEMP_DIR, "raaglist.csv")
raag_info_path = lambda raag_name: os.path.join(TEMP_DIR, f"{raag_name}.pkl")
raw_pickle_path = os.path.join(TEMP_DIR, "raags_temp.pkl")
cleaned_pickle_path = os.path.join(TEMP_DIR, "raags_cleaned.pkl")

## Srape List of Raags

In [ ]:
if (not RECONSTRUCT_FROM_SCRATCH) and (not os.path.exists(raag_list_path)):
    print(f"raaglist.csv not found in {TEMP_DIR}. Would you like to reconstruct it from scratch? (y/n)")
    allow = input().strip().lower()
    if allow == "y":
        RECONSTRUCT_FROM_SCRATCH = True

In [ ]:
if not RECONSTRUCT_FROM_SCRATCH:
    raag_names = []
    refs = []
    with open(raag_list_path, "r") as fp:
        wr = csv.reader(fp, delimiter=",")
        for line in wr:
            raag_names.append(line[0])
            refs.append(line[1])

In [ ]:
if RECONSTRUCT_FROM_SCRATCH:
    ua = input("Get your user agent string from https://www.whatismybrowser.com/detect/what-is-my-user-agent and paste it here: ")
    
    index_url = "https://www.tanarang.com/english/raagIndex_eng.htm"
    resp = requests.get(index_url, headers={"User-Agent": ua})
    if resp.status_code == 200:
        index_soup = BeautifulSoup(resp.text, "html.parser")
        
    # Find the first table
    table = index_soup.find("table")


    # Initialize a list of tuples
    raag_names = []
    refs = []

    # Find all rows within the table and append lists of (name, link)
    rows = table.find_all('tr')
    for row in rows:
        cols = [td.find('a') for td in row.find_all('td')]
        names_links = [(a_tag.text.strip(), a_tag.get('href')) for a_tag in cols]
        for nn, ll in names_links:
            raag_names.append(nn)
            refs.append(ll)
    
    with open(raag_list_path, "w") as fp:
        wr = csv.writer(fp, quoting=csv.QUOTE_ALL)
        for name, ref in zip(raag_names, refs):
            wr.writerow([name, ref])

## Scrape Raag Infos

In [ ]:
def infotable_from_soup(raag_soup) -> pd.DataFrame:
    # Find the first table
    table = raag_soup.find("table")

    # Find all rows within the table
    rows = table.find_all('tr')

    # Initialize a list to store row data
    data = []
    headers = ["info_type", "info"]

    # Loop over the rows (excluding the header row)
    for row in rows:
        cols = row.find_all(['td', 'th'])  # This handles both 'td' and 'th' if 'th' is used within the table body
        cols = [ele.text.strip() for ele in cols]
        data.append(cols)  # Add the data

    # Convert list of row data into a pandas DataFrame
    df = pd.DataFrame(data, columns=headers)
    
    return df

In [ ]:
for name, ref in zip(raag_names, refs):
    
    if (not RECONSTRUCT_FROM_SCRATCH) and (not os.path.exists(raag_info_path(name))):
        print("raag info not found for", name, ". Would you like to fetch it from scratch? (y/n)")
        allow = input().strip().lower()
        if allow == "y":
            RECONSTRUCT_FROM_SCRATCH = True
    
    if not RECONSTRUCT_FROM_SCRATCH:
        print("Loading", name, "from pickle file...")
        df = pd.read_pickle(raag_info_path(name))
    
    if RECONSTRUCT_FROM_SCRATCH:
        url = f"https://www.tanarang.com/english/{ref}"
        resp = requests.get(url, headers={"User-Agent": ua})
        if resp.status_code == 200:
            raag_soup = BeautifulSoup(resp.text, "html.parser")
        df = infotable_from_soup(raag_soup)
        df.to_pickle(raag_info_path(name))

## Parse Scraped Info + Consolidate into a pickle

In [ ]:
with open(raw_pickle_path, "wb") as fp:
    for name in sorted(raag_names):
        df = pd.read_pickle(raag_info_path(name))
        try:
            parsed_raag = TanarangParsedRaag(df, name, verbose=False)
        except:
            print("PROBLEM at", name)
            break
    
        dill.dump(parsed_raag, fp)

In [ ]:
raag_db = {}
with open(raw_pickle_path, "rb") as fp:
    for name in sorted(raag_names):
        raag_db[name] = dill.load(fp)

## Manual Cleanup

In [ ]:
# TODO:
# Get alt names + alt spellings + devanagari

In [ ]:
# TODO: some additions
# Amritavarshini
# Husseini Kanada
# Din Ki Puriya
# Marukauns
# Shuddha Baradi
# Mangal Bhairav
# Shobhawari
# Sundarkali
# Tilang Bahar

In [ ]:
# TODO: clean mukhyangas

Delete some very rare raags with incomplete info

In [ ]:
del raag_db["Shobhawari"]
del raag_db["Suha Sughrai"]
del raag_db["Sundarkali"]
del raag_db["Tilang Bahar"]

Handle warnings

In [ ]:
raag_db["Basant"].vaadi = SSwar("`", "S")
raag_db["Sundarkauns"].vaadi = SSwar("", "m")
raag_db["Sundarkauns"].samvaadi = SSwar("", "S")
raag_db["Sundarkauns"].prahar = "night 2nd"
raag_db["Sundarkauns"].thaat = "Not Defined"
raag_db["Yaman"].aaroha = TanarangParsedRaag.string_to_swars(",N R G M D N S'")
raag_db["Yaman"].avaroha = TanarangParsedRaag.string_to_swars("S' N D P M G R S ,N R S")
raag_db["Yaman"].vaadi = SSwar("", "G")
raag_db["Yaman"].samvaadi = SSwar("", "N")

Typos

In [ ]:
raag_db["Poorvi"].thaat = "Poorvi"

Save Cleaned

In [ ]:
with open(cleaned_pickle_path, "wb") as fp:
    for rd in raag_db.values():
        rd.df = None
        rd.verbose = None
        dill.dump(rd, fp)

## Reading the DB

In [ ]:
with open(cleaned_pickle_path, "rb") as fp:
    raag_db = {}
    while True:
        try:
            rd = dill.load(fp)
            raag_db[rd.name] = rd
        except EOFError:
            break

## Export to JSON

Wraps each cleaned raag's fields under a `"tanarang"` key — leaving room for other
independently-sourced versions (e.g. a future `"empirical"` version) to sit
alongside it under the same raag name — and writes `tanarang.json`, the file
`libmogra/raagdb/__init__.py` loads at import time as `lm.raagdb`.

In [ ]:
def make_str(ss):
    try:
        rr = ss.__str__()
    except:
        rr = str(ss)
    assert type(rr) == str
    return rr

raag_dict_s = {}
for name, rd in raag_db.items():
    rd_dict = rd.__dict__
    rd_dict["aaroha"] = [make_str(ss) for ss in rd_dict["aaroha"]]
    rd_dict["avaroha"] = [make_str(ss) for ss in rd_dict["avaroha"]]
    rd_dict["mukhyanga"] = [[make_str(ss) for ss in mm] for mm in rd_dict["mukhyanga"]]
    rd_dict["aarohi_nyas"] = [make_str(ss) for ss in rd_dict["aarohi_nyas"]]
    rd_dict["avarohi_nyas"] = [make_str(ss) for ss in rd_dict["avarohi_nyas"]]
    rd_dict["vaadi"] = make_str(rd_dict["vaadi"])
    rd_dict["samvaadi"] = make_str(rd_dict["samvaadi"])
    raag_dict_s[name] = rd_dict

In [ ]:
def export(cleaned: dict, out_path="tanarang.json"):
    raag_db = {name.lower(): entry for name, entry in cleaned.items()}
    with open(out_path, "w") as fp:
        json.dump(raag_db, fp, indent=2, ensure_ascii=False)
        fp.write("\n")

In [ ]:
export(raag_dict_s, out_path="tanarang.json")